# 03 — Model Training & Cross-Validation

**Goal:** Train and evaluate two models:
1. **Logistic Regression** — linear baseline with `class_weight='balanced'`
2. **CatBoost** — production model with native categorical support

Then tune CatBoost hyperparameters with **Optuna** and find the optimal classification threshold via precision-recall curve.

> **CV note:** The original production pipeline used a hybrid Sliding + Expanding Window temporal scheme across 11 monthly snapshots. This dataset is cross-sectional (no time dimension), so we use **Stratified K-Fold** (5 folds) which preserves class balance across splits. The CatBoost Optuna tuning script (`run_optuna.py`) used multi-GPU parallelism on 4× NVIDIA A16 cards in production.

In [ ]:
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from catboost import CatBoostClassifier, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost available.')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed. Run: pip install catboost')

sns.set_theme(style='whitegrid')

# Load feature contract
with open('../notebooks/feature_lists.json', 'r') as f:
    contract = json.load(f)

print(f'LogReg features:  {len(contract["X_logreg_cols"])}')
print(f'CatBoost features: {len(contract["X_catboost_cols"])}')

## Part 1 — Logistic Regression Baseline

In [ ]:
df_lr = pd.read_parquet('../data/interim/df_logreg.parquet')

X_lr = df_lr[contract['X_logreg_cols']]
y    = df_lr['Churn']

X_lr = X_lr.select_dtypes(include=[np.number]) # 130726 fix

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_roc_aucs, lr_pr_aucs = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_lr, y)):
    X_train, X_val = X_lr.iloc[train_idx], X_lr.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx],    y.iloc[val_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
    lr.fit(X_train_s, y_train)

    probs = lr.predict_proba(X_val_s)[:, 1]
    roc = roc_auc_score(y_val, probs)
    pr  = average_precision_score(y_val, probs)
    lr_roc_aucs.append(roc)
    lr_pr_aucs.append(pr)

    print(f'Fold {fold+1} | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}')

print(f'\nLogReg Mean | ROC-AUC: {np.mean(lr_roc_aucs):.4f} ± {np.std(lr_roc_aucs):.4f} | PR-AUC: {np.mean(lr_pr_aucs):.4f} ± {np.std(lr_pr_aucs):.4f}')

## Part 2 — CatBoost with Stratified CV

In [ ]:
if not CATBOOST_AVAILABLE:
    print('Skipping CatBoost section. Install with: pip install catboost')
else:
    df_cb = pd.read_parquet('../data/interim/df_catboost.parquet')

    X_cb = df_cb[contract['X_catboost_cols']]
    y_cb = df_cb['Churn']

    # Automatic definition of categorical features
    cat_features = list(X_cb.select_dtypes(include=['object', 'string']).columns) # 130726 fix

    # Ensure string type for categoricals
    X_cb = X_cb.copy()
    for col in cat_features:
        X_cb[col] = X_cb[col].fillna('Unknown').astype(str) # 130726 fix

    cb_roc_aucs, cb_pr_aucs = [], []

    # Tuned hyperparameters from Optuna search (see run_optuna.py)
    cb_params = {
        'iterations':         1100,
        'learning_rate':      0.0339,
        'depth':              7,
        'l2_leaf_reg':        5,
        'random_strength':    3.6568,
        'bagging_temperature': 0.2143,
        'auto_class_weights': 'SqrtBalanced',  # Better than scale_pos_weight for moderate imbalance
        'eval_metric':        'AUC',
        'random_seed':        42,
        'verbose':            0
    }

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb, y_cb)):
        X_train, X_val = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
        y_train, y_val = y_cb.iloc[train_idx], y_cb.iloc[val_idx]

        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        val_pool   = Pool(X_val,   y_val,   cat_features=cat_features)

        model = CatBoostClassifier(**cb_params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)

        probs = model.predict_proba(val_pool)[:, 1]
        roc = roc_auc_score(y_val, probs)
        pr  = average_precision_score(y_val, probs)
        cb_roc_aucs.append(roc)
        cb_pr_aucs.append(pr)

        print(f'Fold {fold+1} | ROC-AUC: {roc:.4f} | PR-AUC: {pr:.4f}')

    print(f'\nCatBoost Mean | ROC-AUC: {np.mean(cb_roc_aucs):.4f} ± {np.std(cb_roc_aucs):.4f} | PR-AUC: {np.mean(cb_pr_aucs):.4f} ± {np.std(cb_pr_aucs):.4f}')

## Part 3 — Final Model + Threshold Optimization

With class imbalance, the default 0.5 threshold is suboptimal.
We find the threshold that maximizes **F1** on the held-out fold.
This mirrors the production pipeline's threshold tuning (0.18 at 27:1 imbalance).

In [ ]:
if not CATBOOST_AVAILABLE:
    print('Skipping. Install catboost first.')
else:
    # Train final model on 80% of data, evaluate threshold on 20%
    from sklearn.model_selection import train_test_split

    X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
        X_cb, y_cb, test_size=0.2, stratify=y_cb, random_state=42
    )

    train_pool_f = Pool(X_train_f, y_train_f, cat_features=cat_features)
    test_pool_f  = Pool(X_test_f,  y_test_f,  cat_features=cat_features)

    final_model = CatBoostClassifier(**cb_params)
    final_model.fit(train_pool_f, eval_set=test_pool_f, use_best_model=True)

    test_probs = final_model.predict_proba(test_pool_f)[:, 1]

    # Precision-recall curve to find optimal F1 threshold
    precision, recall, thresholds = precision_recall_curve(y_test_f, test_probs)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    optimal_idx       = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]

    print(f'Optimal threshold (max F1): {optimal_threshold:.3f}')
    print(f'At this threshold — Precision: {precision[optimal_idx]:.3f}, Recall: {recall[optimal_idx]:.3f}, F1: {f1_scores[optimal_idx]:.3f}')

    # Full report at optimal threshold
    y_pred_opt = (test_probs >= optimal_threshold).astype(int)
    print(f'\nClassification Report at threshold={optimal_threshold:.3f}:')
    print(classification_report(y_test_f, y_pred_opt, target_names=['Retained', 'Churned']))

## Part 4 — Visualization & Feature Importance

In [ ]:
if not CATBOOST_AVAILABLE:
    print('Skipping visualization.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. Precision-Recall curve
    ax = axes[0]
    ax.plot(recall, precision, color='#3498db', lw=2, label=f'CatBoost (PR-AUC={average_precision_score(y_test_f, test_probs):.3f})')
    ax.axvline(recall[optimal_idx], color='#e74c3c', linestyle='--', alpha=0.7, label=f'Optimal threshold={optimal_threshold:.3f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Top feature importances
    ax = axes[1]
    fi = pd.Series(
        final_model.get_feature_importance(),
        index=contract['X_catboost_cols']
    ).sort_values(ascending=True).tail(15)

    fi.plot(kind='barh', ax=ax, color='#2ecc71', edgecolor='white')
    ax.set_title('Top 15 Feature Importances (CatBoost)')
    ax.set_xlabel('Importance Score')

    plt.tight_layout()
    plt.savefig('../reports/03_model_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save final model
    final_model.save_model('../models/catboost_telco_churn_v1.cbm')
    print('Model saved → ../models/catboost_telco_churn_v1.cbm')
    print('\nFinished 03_model_training.ipynb')